# Proyecto: Clasificación de reciclaje con MobileNetV2

Entrenar un modelo con las clases:

- Cardboard
- glass
- metal
- paper
- plastic
- trash

Al final guardara el modelo y las clases para usarlo en Streamlit.

In [ ]:
!pip install tensorflow numpy pillow matplotlib

In [ ]:
from kagglehub import dataset_download
dataset_path = dataset_download("saumyamohandas/garbage-classification-image-dataset")
print("Ruta del dataset: ", dataset_path)

In [ ]:
import os

for folder in os.listdir(dataset_path):
  print(folder)

In [ ]:
import tensorflow as tf

train_data_path = os.path.join(dataset_path, 'dataset', 'Training')
train_ds = tf.keras.preprocessing.image_dataset_from_directory(
    train_data_path,
    image_size=(224, 224),
    batch_size=32
)

class_names = train_ds.class_names
print(class_names)

In [ ]:
import matplotlib.pyplot as plt

for images, labels in train_ds.take(1):
  plt.imshow(images[0].numpy().astype("uint8"))
  plt.title(class_names[labels[0]])
  plt.axis("off")

In [ ]:
# importar mas librerias
import json
import os
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
from PIL import Image

In [ ]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
SEED = 42
OUTPUT_DIR = Path("modelo_reciclaje_mobilenet")
OUTPUT_DIR.mkdir(exist_ok=True)

# definir DATA_DIR AL PATH DE KAGGLEHUB DOWNLOAD
Data_DIR = os.path.join(dataset_path, 'dataset')

train_data_path = os.path.join(Data_DIR, 'Training')
val_data_path = os.path.join(Data_DIR, 'Testing')

train_ds = tf.keras.utils.image_dataset_from_directory(
    train_data_path,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    seed=SEED
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    val_data_path,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    seed=SEED
)

class_names = train_ds.class_names
print('Clases detectadas: ',class_names)

In [ ]:
# hacer una matriz de 3x3
plt.figure(figsize=(10, 8))
for images, labels in train_ds.take(1):
  for i in range(min(9, len(images))):
    ax = plt.subplot(3, 3, i + 1)
    plt.imshow(images[i].numpy().astype('uint8'))
    plt.title(class_names[labels[i]])
    plt.axis('off')

plt.tight_layout()
plt.show()

Este fragmento de código prepara los conjuntos de datos de entrenamiento (train_ds) y validación (val_ds) para ser utilizados por un modelo MobileNetV2 en TensorFlow. Primero, AUTOTUNE = tf.data.AUTOTUNE permite que TensorFlow optimice automáticamente el uso de recursos al cargar los datos. Luego, preprocess_input aplica el preprocesamiento específico de MobileNetV2, transformando los valores de los píxeles de las imágenes al formato esperado por esta red neuronal. Con map(), cada imagen (x) se convierte al tipo float32 mediante tf.cast() y posteriormente se preprocesa, mientras que las etiquetas (y) permanecen sin cambios. Finalmente, prefetch(AUTOTUNE) carga los siguientes lotes de datos en segundo plano mientras el modelo está entrenando, mejorando el rendimiento y reduciendo los tiempos de espera durante el entrenamiento.

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

preprocess_input = tf.keras.applications.mobilenet_v2.preprocess_input

train_ds = train_ds.map(lambda x, y: (preprocess_input(tf.cast(x, tf.float32)), y)).prefetch(AUTOTUNE)
val_ds = val_ds.map(lambda x, y: (preprocess_input(tf.cast(x, tf.float32)), y)).prefetch(AUTOTUNE)

##Crear un modelo de clasificación de imágenes basado en MobileNetV2 preentrenada en ImageNet.

- La red base se mantiene congelada para aprovechar sus características aprendidas, y se agregan nuevas capas para clasificar las imágenes en las categorías definidas.

- Finalmente, el modelo se compila usando el optimizador Adam y la métrica de precisión (accuracy) para su entrenamiento y evaluación.

In [ ]:
#Crear el modelo MobileNetV2
base_model = tf.keras.applications.MobileNetV2(
    input_shape=IMG_SIZE + (3,),
    include_top=False,
    weights="imagenet"
)
base_model.trainable = False

inputs = tf.keras.Input(shape=IMG_SIZE + (3,))
x = base_model(inputs, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.2)(x)
outputs = tf.keras.layers.Dense(len(class_names), activation="softmax")(x)

model = tf.keras.Model(inputs, outputs)

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

##Entrenar

Para probar rápido usa EPOCHS = 1.

In [ ]:
#Earlystopping: detiene el entrenamiento cuando ve que ya no aprende
#Si con 2 EPOCH consecutivas no aprende el modelo, lo va a detener.
EPOCHS = 1

callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=2,
        restore_best_weights=True
    )
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks,
)

In [ ]:
# evaluamos con 4 decimales
loss, acc = model.evaluate(val_ds, verbose=0)
print(f"Loss: {loss:.4f}")
print(f"Accuracy: {acc:.4f}")

In [ ]:
model.save(OUTPUT_DIR / "waste_mobilenet.h5")
#(HDF5 - Hierarchical Data Format version 5)
model.save(OUTPUT_DIR / "waste_mobilenet.keras")

with open(OUTPUT_DIR / "class_names.json", "w", encoding="utf-8") as f:
    json.dump(class_names, f, ensure_ascii=False, indent=2)

print("Modelo guardado en:", OUTPUT_DIR / "waste_mobilenet.h5")
print("Modelo guardado en:", OUTPUT_DIR / "waste_mobilenet.keras")
print("Clases guardadas en:", OUTPUT_DIR / "class_names.json")

In [ ]:
import tensorflow as tf

test_model = tf.keras.models.load_model(
    OUTPUT_DIR / "waste_mobilenet.h5",
    compile=False
)

print("Modelo cargado correctamente")
test_model.summary()

In [ ]:
LABELS_ES= {
    "cardboard": "Cartón",
    "glass": "Vidrio",
    "metal": "Metal",
    "paper": "Papel",
    "plastic": "Plástico",
    "trash": "Basura",
}

def predict_image(img_path_param):
    img = Image.open(img_path_param).convert("RGB").resize(IMG_SIZE)
    arr = np.array(img, dtype=np.float32) / 255.0
    arr = np.expand_dims(arr, axis=0)
    preds = test_model.predict(arr, verbose=0)[0]
    top3_idx = np.argsort(preds)[-3:][::-1]
    for idx in top3_idx:
        raw = class_names[idx]
        print(f"{LABELS_ES.get(raw, raw)}: {preds[idx]*100:.2f}%")
    return class_names[np.argmax(preds)]

# Probar con una imagen del dataset de validación
for images, labels in val_ds.take(1):
    test_img_path = "test_sample.jpeg"
    Image.fromarray(images[0].numpy().astype("uint8")).save(test_img_path)
    result = predict_image(test_img_path)
    plt.imshow(Image.open(test_img_path))
    plt.title(f"Predicción: {LABELS_ES.get(result, result)}")
    plt.axis("off")
    plt.show()
    break